# 01 · Data preparation

This notebook turns the two raw archives into windowed tensors that share one signal
representation:

* one 3-axis accelerometer at the waist (SisFall) or lower trunk (Daphnet);
* axes rotated into an anatomical frame, vertical (V), antero-posterior (AP), medio-lateral (ML), plus the magnitude |a|;
* units in g, sampling rate 64 Hz, 2 s windows.

Comparing a waist sensor in one dataset with an ankle or thigh sensor in the other would mix a
change of placement into the "domain shift". Here the only things allowed to differ between the
domains are the population and the event being detected.

In [1]:
import json
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed

sys.path.insert(0, str(Path.cwd().parent))
from src import config as C
from src import data as D
from src.transforms import RobustScaler

pd.set_option("display.width", 140)
print("windows of", C.WIN, "samples at", C.FS, "Hz; evaluation stride", C.STRIDE, "fine stride", C.FINE_STRIDE)

windows of 128 samples at 64 Hz; evaluation stride 64 fine stride 16


## 1. Extract the raw archives

The zips are copied untouched into `data/raw/archives/` and extracted next to it. Nothing in
`data/raw` is modified afterwards.

In [2]:
ARCH = C.DATA_RAW / "archives"
SIS_DIR = C.DATA_RAW / "sisfall"
DAP_DIR = C.DATA_RAW / "daphnet"


def extract(zip_path, dest):
    if dest.exists() and any(dest.rglob("*.txt")):
        return
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest)
    # some exports nest a second zip inside the first one
    for inner in list(dest.rglob("*.zip")):
        with zipfile.ZipFile(inner) as zf:
            zf.extractall(inner.parent)


extract(ARCH / "SisFall_dataset.zip", SIS_DIR)
extract(ARCH / "Daphnet_dataset.zip", DAP_DIR)

sis_idx = D.index_sisfall(SIS_DIR)
dap_idx = D.index_daphnet(DAP_DIR)
print(f"SisFall: {len(sis_idx)} trial files, {sis_idx.subject.nunique()} subjects")
print(f"Daphnet: {len(dap_idx)} recordings, {dap_idx.subject.nunique()} subjects")
sis_idx.groupby(["group", "is_fall"]).size().unstack().rename(columns={False: "ADL files", True: "Fall files"})

SisFall: 4505 trial files, 38 subjects
Daphnet: 17 recordings, 10 subjects


is_fall,ADL files,Fall files
group,,
Elderly,898,75
Young,1809,1723


## 2. SisFall: parsing, cleaning and unit conversion

Each line holds nine raw counts: ADXL345 (x, y, z), ITG3200 gyroscope (x, y, z) and MMA8451Q
(x, y, z). We keep the ADXL345 only. The second accelerometer sits on the same board, so it
carries almost the same information, and Daphnet has a single accelerometer per site.
Conversion: `g = count * (2 * 16) / 2**13`.

In [3]:
def load_sisfall_trial(path):
    raw = D.parse_sisfall_file(path)
    n_raw = len(raw)
    acc = raw[:, 0:3] * C.ADXL345_G_PER_COUNT if n_raw else np.empty((0, 3))
    n_missing = int((~np.isfinite(acc)).any(axis=1).sum()) if n_raw else 0
    acc = D.fill_missing(acc) if n_raw else None
    if acc is None:
        return None, {"n_raw": n_raw, "n_missing": n_missing}
    # physical range of the ADXL345 configuration; anything beyond is a corrupted sample
    n_clipped = int((np.abs(acc) > 16).any(axis=1).sum())
    acc = np.clip(acc, -16, 16)
    acc64 = D.resample_to(acc, C.SISFALL_FS, C.FS).astype(np.float32)
    return acc64, {"n_raw": n_raw, "n_missing": n_missing, "n_clipped": n_clipped}


results = Parallel(n_jobs=2, verbose=0)(delayed(load_sisfall_trial)(p) for p in sis_idx.path)
sis_sig = {}
qc = []
for (sig, info), (_, row) in zip(results, sis_idx.iterrows()):
    qc.append({**row.to_dict(), **info, "kept": sig is not None and len(sig) >= C.WIN})
    if sig is not None and len(sig) >= C.WIN:
        sis_sig[row.file] = sig
qc = pd.DataFrame(qc)
qc["duration_s"] = qc.n_raw / C.SISFALL_FS
print("files kept:", qc.kept.sum(), "/", len(qc))
print("samples with missing values:", qc.n_missing.sum(), "in", (qc.n_missing > 0).sum(), "files")
print("samples clipped to +/-16 g:", qc.n_clipped.sum())
qc.groupby("is_fall").duration_s.describe()

files kept: 4505 / 4505
samples with missing values: 0 in 0 files
samples clipped to +/-16 g: 0


,count,mean,std,min,25%,50%,75%,max
is_fall,,,,,,,,
False,2707.0,19.329913,20.897225,9.995,12.0,12.0,12.0,180.00
True,1798.0,14.999205,0.013471,14.455,15.0,15.0,15.0,15.03


## 3. Finding the anatomical axes of the SisFall sensor

Rather than trusting a figure in the dataset paper, the axes are identified from the data:

* **Vertical**: during upright walking (D01-D04) the static component is ~1 g on one axis.
* **AP vs. ML**: after a fall the subject lies still, so gravity ends up on the horizontal axis
  that the fall direction points to. Forward/backward falls (F01, F02, F04, F05, ...) end on the
  AP axis, lateral falls (F03, F06, ...) on the ML axis.

Sign convention for the common frame: an axis reads +1 g when it points upward. Standing:
V = +1 g. Lying prone after a forward fall: AP = -1 g. Lying supine after a backward fall: AP = +1 g.

In [4]:
walk = sis_idx[sis_idx.activity.isin(["D01", "D02", "D03", "D04"])]
walk_means = np.array([sis_sig[f].mean(axis=0) for f in walk.file if f in sis_sig])
print("mean acceleration during walking per raw axis (g):", walk_means.mean(0).round(3))
v_axis = int(np.argmax(np.abs(walk_means.mean(0))))
v_sign = int(np.sign(walk_means.mean(0)[v_axis]))

# SisFall fall codes (dataset README): direction of each fall
# (F06 is a vertical collapse and F07 is damped with the hands on a table, so neither has a
# clear final orientation)
FALL_DIRECTION = {
    "F01": "forward", "F02": "backward", "F03": "lateral", "F04": "forward",
    "F05": "forward", "F06": "vertical", "F07": "other", "F08": "forward",
    "F09": "lateral", "F10": "forward", "F11": "backward", "F12": "lateral",
    "F13": "forward", "F14": "backward", "F15": "lateral",
}
post = []
for _, r in sis_idx[sis_idx.is_fall].iterrows():
    if r.file not in sis_sig:
        continue
    s = sis_sig[r.file]
    post.append({"activity": r.activity, "direction": FALL_DIRECTION[r.activity],
                 **{f"raw_{k}": s[-3 * C.FS:, k].mean() for k in range(3)}})
post = pd.DataFrame(post)
post_summary = post.groupby("direction")[["raw_0", "raw_1", "raw_2"]].agg(lambda x: np.mean(np.abs(x)))
print("mean |acc| over the last 3 s of fall trials, by fall direction (g):")
print(post_summary.round(3))

horiz = [k for k in range(3) if k != v_axis]
fb = post_summary.loc[["forward", "backward"], [f"raw_{k}" for k in horiz]].mean()
ap_axis = horiz[int(np.argmax(fb.values))]
ml_axis = [k for k in horiz if k != ap_axis][0]
fwd_mean = post.loc[post.direction == "forward", f"raw_{ap_axis}"].mean()
# prone after a forward fall -> forward-pointing axis faces the floor -> reads -1 g
ap_sign = int(-np.sign(fwd_mean))
SISFALL_MAP = {"V": (v_axis, v_sign), "AP": (ap_axis, ap_sign), "ML": (ml_axis, 1)}
print("SisFall ADXL345 -> anatomical frame:", SISFALL_MAP)

mean acceleration during walking per raw axis (g): [ 0.006 -0.968 -0.235]


mean |acc| over the last 3 s of fall trials, by fall direction (g):
           raw_0  raw_1  raw_2
direction                     
backward   0.208  0.188  0.832
forward    0.533  0.161  0.713
lateral    0.864  0.133  0.339
other      0.531  0.133  0.646
vertical   0.605  0.152  0.614
SisFall ADXL345 -> anatomical frame: {'V': (1, -1), 'AP': (2, 1), 'ML': (0, 1)}


## 4. SisFall windows and labels

SisFall gives one label per file, not per sample. Calling every window of a fall file a "fall"
would label the walking before the fall and the lying after it as falls. Instead, the impact is located as the maximum of |a| in each fall trial, and a window is
a fall window when it overlaps the interval impact +/- 1 s by at least 1.5 s. Windows that touch
that interval only partially are discarded as ambiguous; windows entirely outside it are kept as
non-fall examples (pre-fall walking, getting up, lying).

In [5]:
split_of = {s: k for k, subs in C.SISFALL_SPLIT.items() for s in subs}
assert set(sis_idx.subject) <= set(split_of), set(sis_idx.subject) - set(split_of)

X_list, meta = [], []
for _, r in sis_idx.iterrows():
    if r.file not in sis_sig:
        continue
    sig = D.to_frame(sis_sig[r.file], SISFALL_MAP)
    starts = D.sliding_windows(len(sig))
    if r.is_fall:
        # ignore the first and last second: the protocol starts and ends at rest,
        # and a bump while setting the logger down must not be taken as the impact
        mag = sig[:, 3].copy()
        mag[: C.FS] = 0
        mag[-C.FS:] = 0
        peak = int(np.argmax(mag))
        labels = D.fall_window_labels(starts, peak)
    else:
        peak = -1
        labels = np.zeros(len(starts), dtype=int)
    split = split_of[r.subject]
    for s, lab in zip(starts, labels):
        if lab < 0:
            continue
        coarse = bool(s % C.STRIDE == 0)
        # Training keeps the 1 s grid plus every fall window on the 0.25 s grid.
        # Validation and test use the 1 s grid only, as a deployed detector would.
        if not (coarse or (split == "train" and lab == 1)):
            continue
        X_list.append(sig[s:s + C.WIN])
        meta.append({"domain": "SisFall", "subject": r.subject, "group": r.group,
                     "activity": r.activity, "trial": r.trial, "file": r.file,
                     "is_fall_trial": r.is_fall, "start": int(s), "peak": peak,
                     "label": int(lab), "coarse": coarse, "split": split})
X_sis = np.stack(X_list).astype(np.float32)
meta_sis = pd.DataFrame(meta)
del X_list
print(X_sis.shape)

summary_sis = (meta_sis.groupby(["split", "label"]).size().unstack(fill_value=0)
               .rename(columns={0: "non-fall", 1: "fall"}))
summary_sis["subjects"] = meta_sis.groupby("split").subject.nunique()
summary_sis["fall %"] = (100 * summary_sis["fall"] / (summary_sis["fall"] + summary_sis["non-fall"])).round(2)
summary_sis.loc[["train", "val", "test"]]

(73103, 128, 4)


label,non-fall,fall,subjects,fall %
split,,,,
train,44705,4877,25,9.84
val,8738,229,5,2.55
test,14173,381,8,2.62


## 5. Daphnet: trunk accelerometer, annotations and windows

The archive holds both a CSV re-export and the original UCI release; the original
`dataset_fog_release/dataset/*.txt` files (17 recordings) are used.

Columns: time (ms), ankle, thigh and trunk accelerometers (forward, vertical, lateral; mg),
annotation (0 = outside the experiment, 1 = no freeze, 2 = freeze). Only the trunk sensor is
used; it sits on the lower back, the closest site to the SisFall waist logger. Samples
annotated 0 are removed, and windows are cut inside each contiguous annotated segment so that
no window joins two pieces of recording that were not adjacent in time.

In [6]:
dap_frames = {r.file: D.parse_daphnet_file(r.path) for _, r in dap_idx.iterrows()}
dt = np.concatenate([np.diff(df.t_ms.values) for df in dap_frames.values()])
# timestamps are integer ms, so consecutive differences alternate between 15 and 16 ms
print(f"Daphnet sampling interval: mean {np.mean(dt):.3f} ms -> {1000 / np.mean(dt):.2f} Hz")
nan_rows = sum(int(df.isna().any(axis=1).sum()) for df in dap_frames.values())
print("rows with missing values:", nan_rows)

trunk_vert_mean = np.mean([df.loc[(df.annotation == 1) & (df.trunk_vert != 0), "trunk_vert"].mean()
                           for df in dap_frames.values()])
print(f"mean trunk vertical acceleration during walking: {trunk_vert_mean:.0f} mg")
DAPHNET_MAP = {"V": (1, int(np.sign(trunk_vert_mean))), "AP": (0, 1), "ML": (2, 1)}
print("Daphnet trunk -> anatomical frame:", DAPHNET_MAP)

X_list, meta = [], []
dap_qc = []
for _, r in dap_idx.iterrows():
    df = dap_frames[r.file]
    trunk = df[["trunk_fwd", "trunk_vert", "trunk_lat"]].to_numpy(dtype=np.float64) / 1000.0
    # A trunk reading of exactly (0, 0, 0) mg is physically impossible (gravity alone gives
    # ~1000 mg): it marks a sensor dropout. Those samples are treated as gaps.
    dropout = (trunk == 0).all(axis=1)
    trunk[dropout] = np.nan
    trunk = D.fill_missing(trunk)
    ann = df.annotation.to_numpy()
    sig = D.to_frame(trunk, DAPHNET_MAP)
    segs = D.contiguous_segments((ann > 0) & ~dropout)
    dap_qc.append({"file": r.file, "subject": r.subject, "duration_min": len(df) / C.FS / 60,
                   "annotated_min": (ann > 0).sum() / C.FS / 60, "fog_min": (ann == 2).sum() / C.FS / 60,
                   "dropout_annotated_min": (dropout & (ann > 0)).sum() / C.FS / 60,
                   "segments": len(segs)})
    for a, b in segs:
        for s in D.sliding_windows(b - a, stride=C.STRIDE):
            w_ann = ann[a + s: a + s + C.WIN]
            frac = float((w_ann == 2).mean())
            X_list.append(sig[a + s: a + s + C.WIN])
            meta.append({"domain": "Daphnet", "subject": r.subject, "group": "Parkinson",
                         "activity": "FoG" if frac >= 0.5 else "Walk", "trial": r.run, "file": r.file,
                         "is_fall_trial": False, "start": int(a + s), "peak": -1,
                         "label": int(frac >= 0.5), "fog_fraction": frac, "coarse": True,
                         "split": "target", "use": True})
X_dap = np.stack(X_list).astype(np.float32)
meta_dap = pd.DataFrame(meta)
del X_list
dap_qc = pd.DataFrame(dap_qc)
print(X_dap.shape, "FoG windows:", meta_dap.label.sum(), f"({100 * meta_dap.label.mean():.2f}%)")
dap_qc.groupby("subject")[["duration_min", "annotated_min", "fog_min", "dropout_annotated_min"]].sum().round(1)

Daphnet sampling interval: mean 15.625 ms -> 64.00 Hz
rows with missing values: 0
mean trunk vertical acceleration during walking: 975 mg
Daphnet trunk -> anatomical frame: {'V': (1, 1), 'AP': (0, 1), 'ML': (2, 1)}


(17410, 128, 4) FoG windows: 1734 (9.96%)


,duration_min,annotated_min,fog_min,dropout_annotated_min
subject,,,,
S01,53.1,31.7,1.7,0.0
S02,42.2,23.6,3.0,0.0
S03,65.9,33.5,4.8,0.0
S04,51.0,34.5,0.0,0.0
S05,54.9,34.8,7.9,0.0
S06,57.3,33.2,2.2,0.0
S07,44.2,26.8,1.4,0.0
S08,35.6,12.8,3.3,0.0
S09,44.9,29.0,4.5,0.0


## 6. Scaling

Robust z-score per channel, `(x - median) / IQR`. Statistics:

* **source**: fitted on SisFall training subjects only (1 s grid, so the duplicated fall windows
  do not weigh on the statistics);
* **target (independent)**: fitted on all Daphnet windows *without labels*. Using unlabeled
  target data this way is legitimate in a zero-shot setting and is the scaling used by default;
* per-subject and source-only variants are built later as ablations.

The unscaled windows (in g) are saved as well, because training-time augmentation has to be done
in physical units.

In [7]:
m_tr = (meta_sis.split == "train") & meta_sis.coarse
src_scaler = RobustScaler().fit(X_sis[m_tr.values])
tgt_scaler = RobustScaler().fit(X_dap)
scalers = {"source": src_scaler.state(), "target": tgt_scaler.state(), "channels": C.CHANNELS,
           "sisfall_axis_map": {k: list(v) for k, v in SISFALL_MAP.items()},
           "daphnet_axis_map": {k: list(v) for k, v in DAPHNET_MAP.items()}}
print(json.dumps({k: scalers[k] for k in ("source", "target")}, indent=1))

{
 "source": {
  "center": [
   0.8988603949546814,
   -0.10589929670095444,
   0.020852848887443542,
   0.9956396222114563
  ],
  "scale": [
   0.7113673686981201,
   0.5162761211395264,
   0.18470638990402222,
   0.11469870805740356
  ]
 },
 "target": {
  "center": [
   0.9800000190734863,
   0.04800000041723251,
   -0.01899999938905239,
   1.0157170295715332
  ],
  "scale": [
   0.08600002527236938,
   0.2409999966621399,
   0.26100000739097595,
   0.04014146327972412
  ]
 }
}


## 7. Save arrays, metadata and PyTorch tensors

In [8]:
out = C.DATA_PROCESSED
np.save(out / "sisfall_X_g.npy", X_sis)
np.save(out / "daphnet_X_g.npy", X_dap)
meta_sis.to_csv(out / "sisfall_meta.csv", index=False)
meta_dap.to_csv(out / "daphnet_meta.csv", index=False)
with open(out / "scalers.json", "w") as fh:
    json.dump(scalers, fh, indent=2)

for split in ("train", "val", "test"):
    m = (meta_sis.split == split).values
    torch.save({"X": torch.from_numpy(src_scaler.transform(X_sis[m])),
                "X_g": torch.from_numpy(X_sis[m]),
                "y": torch.from_numpy(meta_sis.label.values[m].astype(np.int64)),
                "subject": meta_sis.subject.values[m].tolist(), "channels": C.CHANNELS},
               out / f"sisfall_{split}.pt")
torch.save({"X": torch.from_numpy(tgt_scaler.transform(X_dap)),
            "X_g": torch.from_numpy(X_dap),
            "y": torch.from_numpy(meta_dap.label.values.astype(np.int64)),
            "subject": meta_dap.subject.tolist(), "channels": C.CHANNELS},
           out / "daphnet_target.pt")

qc.to_csv(C.METRICS / "sisfall_file_qc.csv", index=False)
dap_qc.to_csv(C.METRICS / "daphnet_file_qc.csv", index=False)
summary = pd.concat([
    summary_sis.loc[["train", "val", "test"]].assign(domain="SisFall"),
    pd.DataFrame({"non-fall": [int((meta_dap.label == 0).sum())], "fall": [int(meta_dap.label.sum())],
                  "subjects": [meta_dap.subject.nunique()],
                  "fall %": [round(100 * meta_dap.label.mean(), 2)], "domain": ["Daphnet"]},
                 index=["target"]),
]).rename(columns={"non-fall": "negatives", "fall": "positives", "fall %": "positive %"})
summary.to_csv(C.METRICS / "dataset_summary.csv")
summary

,negatives,positives,subjects,positive %,domain
train,44705,4877,25,9.84,SisFall
val,8738,229,5,2.55,SisFall
test,14173,381,8,2.62,SisFall
target,15676,1734,10,9.96,Daphnet


In [9]:
for p in sorted(out.iterdir()):
    print(f"{p.name:28s} {p.stat().st_size / 1e6:8.1f} MB")

daphnet_X_g.npy                  35.7 MB
daphnet_meta.csv                  1.4 MB
daphnet_target.pt                71.5 MB
scalers.json                      0.0 MB
sisfall_X_g.npy                 149.7 MB
sisfall_meta.csv                  5.1 MB
sisfall_test.pt                  59.8 MB
sisfall_train.pt                203.8 MB
sisfall_val.pt                   36.8 MB
